In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error


In [ ]:


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
file_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(file_path)

In [ ]:
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
print("\nDataFrame Description:")
df.describe()

In [ ]:

plt.hist(df['Delivery_Time'].dropna(), bins=30)
plt.title('Delivery Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# delet the column
df = df.drop(columns=['Order_ID'])

# to ensure it deleted
print(df.columns)

In [ ]:
print("Missing values:")
print(df.isnull().sum())

df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])

df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median())
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].median())

print(df.isnull().sum())

In [ ]:
df = df.drop_duplicates() #delete any duplicates column if there

In [ ]:
df = pd.get_dummies(df, columns=['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'])

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features = df.drop(columns=['Delivery_Time']).columns
df[features] = scaler.fit_transform(df[features])

In [ ]:
#determine data and target
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

In [ ]:

# we use k fold , k = 5
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# intilaization the model
model = RandomForestRegressor(n_estimators=100, random_state=42)

# use MSA
mae_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
mae_scores = -mae_scores

# 5. print the average
print(f"Average MAE across all folds: {mae_scores.mean():.2f}")

In [ ]:

# start training
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

# plot
plt.figure(figsize=(10, 8))
importances.plot(kind='barh', color='teal')
plt.title('Feature Importance - What affects delivery time?')
plt.xlabel('Importance Score')
plt.show()

In [ ]:
# Get the forecast
y_pred = model.predict(X)

# plot
plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, color='orange', alpha=0.7, label='Predicted')
plt.hist(y, bins=30, color='blue', alpha=0.3, label='Actual')
plt.title('Distribution of Predicted vs Actual Delivery Time')
plt.xlabel('Delivery Time (min)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:


kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
cb_model = CatBoostRegressor(verbose=0, random_state=42) # verbose=0 لتقليل المخرجات

mae_list = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf_model.fit(X_train, y_train)
    cb_model.fit(X_train, y_train)

    rf_preds = rf_model.predict(X_test)
    cb_preds = cb_model.predict(X_test)

    final_preds = (rf_preds + cb_preds) / 2

    mae = mean_absolute_error(y_test, final_preds)
    mae_list.append(mae)

print(f"Average Ensemble MAE: {np.mean(mae_list):.4f}")